In [2]:
import joblib
import pandas as pd

df_users = pd.read_parquet("data\\11092026\\users.parquet")
df_movies = pd.read_parquet("data\\11092026\\movies.parquet")
df_interactions = pd.read_parquet("data\\11092026\\interactions.parquet")

encoders = joblib.load("data\\11092026\\encoders.pkl")
metadata = joblib.load("data\\11092026\\metadata.pkl")

In [3]:
df_interactions.head(10)

df_interactions["movie_id"] = (
    encoders["movie_encoder"].inverse_transform(df_interactions["movie_idx"])
)

df_interactions["user_id"] = (
    encoders["user_encoder"].inverse_transform(df_interactions["user_idx"])
)

df_interactions


,user_idx,movie_idx,rating,timestamp,movie_id,user_id
0,0,1104,5.0,978300760,1193,1
1,0,639,3.0,978302109,661,1
2,0,853,3.0,978301968,914,1
3,0,3177,4.0,978300275,3408,1
4,0,2162,5.0,978824291,2355,1
...,...,...,...,...,...,...
1000204,6039,1019,1.0,956716541,1091,6040
1000205,6039,1022,5.0,956704887,1094,6040
1000206,6039,548,5.0,956704746,562,6040
1000207,6039,1024,4.0,956715648,1096,6040


In [4]:
df_full = df_users.merge(df_interactions, on="user_id", how="left")
df_full = df_full.merge(df_movies, on="movie_id", how="left")



In [5]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))

from src.temporal_split import TemporalSplit


### Temporal stats
Данный метод создавался с целью улучшить метрики модели. Подсчет признаков осуществлялся с изменением во времени, но по итогу исследований оказалось что данные признаки не несут особой информации

In [6]:
def temporal_stats(train):

    df = train.copy().sort_values(["movie_id", "timestamp"])
    df["_sq_rating"] = df["rating"]**2

    g = df.groupby("movie_id", sort=False)

    movie_count = g.cumcount()
    previous_sum_rating = g["rating"].cumsum() - df["rating"]
    previous_sq_sum_rating = g["_sq_rating"].cumsum() - df["_sq_rating"]

    df["count_movie_rating"] = movie_count
    df["avg_movie_rating"] = previous_sum_rating / movie_count
    # df["std_movie_rating"] = np.sqrt(
    #     (previous_sq_sum_rating - (previous_sum_rating)**2 / movie_count)
    #     / (movie_count - 1)
    # ).where(movie_count > 1)


    return df.drop(columns="_sq_rating").sort_values(by="user_id")

### Temporal split data
Разбиение данных относительно временной шкалы:
<center>train.timestamp < val.timestamp < test.timestamp</center>

In [40]:
import numpy as np

df_full["rating_binary"] = (df_full["rating"] >= 4).astype(int)
df_full["year"] = (df_full["title"].str.extract(r"\((\d+)\)")).astype(int)

drop_cols = [
    "movie_idx",
    "user_idx",
    "genre",
    "title",
    "timestamp",
    "zip_code",
    "rating",
    "rating_binary",
    "title",
]

cat_features = [
    "user_id",
    "movie_id",
    "gender",
    "age_group",
    "occupation",
    "movie_id"
]

train , val , test = TemporalSplit().split(data=df_full)

#train = temporal_stats(train)

group_train = train["user_id"]
group_val = val["user_id"]

y_train = train["rating_binary"]
y_val = val["rating_binary"]
y_test = test["rating_binary"]

X_train = train.drop(columns=drop_cols)
X_val = val.drop(columns=drop_cols)
X_test = test.drop(columns=drop_cols)

X_train

,user_id,gender,age_group,occupation,movie_id,Action,Adventure,Animation,Children's,Comedy,...,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western,year
0,1,0,1,10,3186,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1999
1,1,0,1,10,1270,0,0,0,0,1,...,0,0,0,0,0,1,0,0,0,1985
2,1,0,1,10,1721,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,1997
3,1,0,1,10,1022,0,0,1,1,0,...,0,0,1,0,0,0,0,0,0,1950
4,1,0,1,10,2340,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,1998
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
797753,6040,1,25,6,2645,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,1958
797754,6040,1,25,6,1449,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,1996
797755,6040,1,25,6,3504,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,1976
797756,6040,1,25,6,2303,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,1975


### Sampled data
Была предпринята попытка семплировать данные для упрощения обучения и улучшение метрик, к каким то улучшениям это не привело + решает немного не ту задачу которая нам требуется

In [ ]:
import random
random.seed(42)

df = pd.DataFrame()
user_id = 1

for user_id in val["user_id"].unique():

    pos_table = (
        val[
            (val["user_id"] == user_id) &
            (val["rating_binary"] == 1)
        ][["user_id", "movie_id", "rating_binary"]]
    )

    watched = set(
        train[
            train["user_id"] == user_id
        ]["movie_id"]
    )

    neg_list = list(set(df_movies["movie_id"])
            - watched
            - set(pos_table["movie_id"]))

    negative = random.sample(
        neg_list,
        k=min(len(neg_list), len(pos_table) * 5)
    )

    user_mask = val["user_id"] == user_id
    neg_mask = df_movies["movie_id"].isin(negative) 

    neg_table = df_movies.loc[neg_mask, ["movie_id"]].copy()
    neg_table["user_id"] = user_id
    neg_table["rating_binary"] = 0

    table = pd.concat([pos_table, neg_table])

    df = pd.concat([df, table])

df

,user_id,movie_id,rating_binary
0,1,595,1
3,1,588,1
4,1,1,1
103,1,105,0
359,1,363,0
...,...,...,...
1899,6040,1968,0
2076,6040,2145,0
3235,6040,3304,0
3590,6040,3659,0


### Fit model
Метод для обучения и сохранения модели в файл

In [ ]:
from catboost import CatBoostRanker
import joblib

def fit(model_name="test"):

    model = CatBoostRanker(
        loss_function="YetiRank",
        eval_metric="NDCG:top=10",
        iterations=500,
        learning_rate=0.01,
        depth=5,
        random_seed=42,
        verbose=50
    )

    model.fit(
        X_train,
        y_train,
        group_id=group_train,
        cat_features=cat_features
    )

    joblib.dump(
        model,
        f"data\\models\\{model_name}.pkl"
    )

Использовался для подсчета агрегированных состояний 

In [16]:
def movie_stats(history):
    table_movie = (
        history
        .groupby("movie_id")
        .agg(
            avg_movie_rating=("rating", "mean"),
            count_movie_rating=("rating", "count"),
            #std_movie_rating=("rating", "std"),
        )
        .reset_index()
    )

    # table_user = (
    #         history
    #         .groupby("user_id")
    #         .agg(
    #             avg_user_rating=("rating", "mean"),
    #             count_user_rating=("rating", "count"),
    #         )
    #         .reset_index()
    #     )

    return table_movie#, table_user


### Evaluation method
Метод для оценки качества модели

In [49]:
ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))

from src.metrics.metrics import (
    recall_at_k,
    precision_at_k,
    ndcg_at_k,
)
import numpy as np

drop_cols = [
    "genre",
    "zip_code",
    "title",
]

def eval(
        model,
        train,
        val,
        df_movie,
        df_users,
        feature_collumns,
        k=10,
        model_name="model"
):


    train_seen = (
        train
        .groupby("user_id")["movie_id"]
        .agg(set)
        .to_dict()
    )

    recalls = []
    precisions = []
    ndcgs = []


    movie_features = df_movie.copy()

    movie_features["year"] = (
        movie_features["title"]
        .str.extract(r"\((\d{4})\)")
        .astype("Int64")
    )

    movie_features = movie_features.drop(columns=["genre", "title"])

    user_features = df_users.copy()
    user_features = user_features.drop(columns="zip_code")

    for user_id in val.user_id.unique():

        watched = train_seen.get(user_id, set())

        candidates = movie_features[
            ~movie_features["movie_id"].isin(watched)
        ]

        candidates["user_id"] = user_id

        candidates = candidates.merge(user_features, on="user_id", how="left")

        candidates = candidates[feature_collumns]

        scores = model.predict(candidates)

        top_indices = np.argsort(scores)[::-1][:k]

        recommended = (
            candidates["movie_id"]
            .iloc[top_indices]
            .tolist()
        )

        val_per_user = val[
            val["user_id"] == user_id
        ]

        relevant = val_per_user.loc[
            val_per_user["rating"] >= 4,
            "movie_id"
        ]

        recall = recall_at_k(relevant, recommended)
        precision = precision_at_k(relevant, recommended)
        ndcg = ndcg_at_k(relevant, recommended)

        recalls.append(recall)
        precisions.append(precision)
        ndcgs.append(ndcg)

    metrics = {
        "recall": np.mean(recalls),
        "precision": np.mean(precisions),
        "ndcg": np.mean(ndcgs)
    }

    joblib.dump(
        metrics,
        f"data\\metrics\\{model_name}.pkl"
    )

    return metrics


In [43]:
import joblib

model = joblib.load("data\\models\\baseline_final.pkl")
model

CatBoostRanker(depth=5, eval_metric='NDCG:top=10', iterations=500, learning_rate=0.01, loss_function='YetiRank', random_seed=42, verbose=50)

In [41]:

model = fit("baseline_final")

Groupwise loss function. OneHotMaxSize set to 10
0:	total: 725ms	remaining: 6m 1s
50:	total: 34.7s	remaining: 5m 5s
100:	total: 1m 9s	remaining: 4m 34s
150:	total: 1m 44s	remaining: 4m 2s
200:	total: 2m 19s	remaining: 3m 27s
250:	total: 2m 54s	remaining: 2m 52s
300:	total: 3m 29s	remaining: 2m 18s
350:	total: 4m 4s	remaining: 1m 43s
400:	total: 4m 40s	remaining: 1m 9s
450:	total: 5m 15s	remaining: 34.3s
499:	total: 5m 50s	remaining: 0us


In [ ]:
res = eval(model, train, val, df_movies, df_users, X_train.columns, model_name="baseline_final")

### Result

Baseline:
- Recall@10: 0.02356
- Precision@10: 0.01868
- NDCG@10: 0.02543

+ temporal movie features:
- Recall@10: 0.01850
- Precision@10: 0.01710
- NDCG@10: 0.02142

Вывод: в текущей реализации агрегатные признаки ухудшают результат.

### Model with candidat generation

In [1]:
import sys
from pathlib import Path
import joblib
import pandas as pd

ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))

from src.temporal_split import TemporalSplit
from src.models.recommender import Recommender
from src.models.matrix_factorization import FactorizationRecommender
from src.models.ranker import Ranker
from src.metrics.evaluation import eval_pop, eval_factorization

In [2]:
df_users = pd.read_parquet("data\\11092026\\users.parquet")
df_movies = pd.read_parquet("data\\11092026\\movies.parquet")
df_interactions = pd.read_parquet("data\\11092026\\interactions.parquet")

encoders = joblib.load("data\\11092026\\encoders.pkl")
metadata = joblib.load("data\\11092026\\metadata.pkl")

n_movies = metadata["n_movies"]
n_users = metadata["n_users"]

df_interactions["movie_id"] = (
    encoders["movie_encoder"].inverse_transform(df_interactions["movie_idx"])
)

df_interactions["user_id"] = (
    encoders["user_encoder"].inverse_transform(df_interactions["user_idx"])
)

df_full = df_users.merge(df_interactions, on="user_id", how="left")
df_full = df_full.merge(df_movies, on="movie_id", how="left")
df_full["rating_binary"] = (df_full["rating"] >= 4).astype(int)
df_full["year"] = (df_full["title"].str.extract(r"\((\d+)\)")).astype(int)

In [3]:
drop_cols = [
    "movie_idx",
    "user_idx",
    "genre",
    "title",
    "timestamp",
    "zip_code",
    "rating",
    "rating_binary",
    "title",
]

cat_features = [
    "user_id",
    "movie_id",
    "gender",
    "age_group",
    "occupation",
    "movie_id"
]

train , val , test = TemporalSplit().split(data=df_full)

group_train = train["user_id"]

y_train = train["rating_binary"]
y_val = val["rating_binary"]
y_test = test["rating_binary"]



In [ ]:
popularity_model = Recommender()\

popularity_model.fit(df_interactions)
pop_metrics = eval_pop(model=popularity_model, train=train, val=val, interactions=df_interactions)


{'recall@10': np.float64(0.029265108946395568), 'precision@10': np.float64(0.02665267576075551), 'NDCG@10': np.float64(0.03478565010597663)}


In [ ]:
factorization_model = FactorizationRecommender(
    n_users=n_users, 
    n_movies=n_movies, 
    n_factor=10, 
    lr=0.01, 
    reg=0.01, 
    epochs=20
)

factorization_model.fit(train=train)
factorization_metrics = eval_factorization(model = factorization_model, train=train, val=val)


In [6]:
catboost_model = Ranker(
    loss_function="YetiRank",
    eval_metric="NDCG:top=10",
    iterations=500,
    learning_rate=0.01,
    depth=5,
    random_seed=42,
    verbose=50
)

catboost_model.fit(X=X_train, y=y_train, group=group_train, features=cat_features)

Groupwise loss function. OneHotMaxSize set to 10
0:	total: 972ms	remaining: 8m 4s


KeyboardInterrupt: 

### Hyperparameter tuning

In [5]:
X_train = train.drop(columns=drop_cols)
X_val = val.drop(columns=drop_cols)
X_test = test.drop(columns=drop_cols)